# QTS System Demo

End-to-end walkthrough of the Quantitative Trading System:

1. **Data** — generate synthetic OHLCV bars via `MockBarAdapter`
2. **Signals** — compute technical indicators (RSI, MACD, BB, ATR, Momentum)
3. **Regime** — HMM-based volatility regime detection
4. **Sentiment** — VADER analysis + multi-source fusion
5. **Alpha** — combined alpha signal from all sources
6. **Backtest** — run strategies and compare performance

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure src is on the path
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

%matplotlib inline
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Generate Synthetic Market Data

In [ ]:
from qts.data.market.mock_adapter import MockBarAdapter

SYMBOL = "BTCUSDT"
N_BARS = 500
SEED = 42

adapter = MockBarAdapter(seed=SEED, n_bars=N_BARS, initial_price=50_000.0)
bars = await adapter.get_historical_bars(SYMBOL, start="", end="")

# Build a DataFrame for easy plotting
df = pd.DataFrame([
    {"timestamp": b.timestamp, "open": b.open, "high": b.high,
     "low": b.low, "close": b.close, "volume": b.volume}
    for b in bars
]).set_index("timestamp")

print(f"{len(bars)} bars generated | Price range: ${df['low'].min():,.0f} – ${df['high'].max():,.0f}")
df.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(df.index, df["close"], linewidth=0.8, color="#2196F3")
ax1.fill_between(df.index, df["low"], df["high"], alpha=0.15, color="#2196F3")
ax1.set_ylabel("Price (USD)")
ax1.set_title(f"{SYMBOL} — Synthetic Price Series ({N_BARS} bars)")

ax2.bar(df.index, df["volume"], width=0.0005, color="#78909C", alpha=0.7)
ax2.set_ylabel("Volume")
ax2.set_xlabel("Time")

plt.tight_layout()
plt.show()

## 2. Technical Indicators

In [ ]:
from qts.signals.indicators import (
    compute_atr,
    compute_bb_position,
    compute_bollinger_bands,
    compute_macd,
    compute_momentum,
    compute_rsi,
)

closes = np.array([b.close for b in bars], dtype=np.float64)
highs = np.array([b.high for b in bars], dtype=np.float64)
lows = np.array([b.low for b in bars], dtype=np.float64)

rsi = compute_rsi(closes)
macd_line, macd_signal, macd_hist = compute_macd(closes)
bb_upper, bb_middle, bb_lower = compute_bollinger_bands(closes)
atr = compute_atr(highs, lows, closes)
momentum = compute_momentum(closes)

print(f"Latest RSI: {rsi[~np.isnan(rsi)][-1]:.1f}")
print(f"Latest MACD histogram: {macd_hist[~np.isnan(macd_hist)][-1]:.4f}")
print(f"Latest ATR: {atr[~np.isnan(atr)][-1]:.2f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
idx = df.index

# Price + Bollinger Bands
axes[0].plot(idx, closes, linewidth=0.8, color="#2196F3", label="Close")
axes[0].plot(idx, bb_upper, linewidth=0.5, color="#E91E63", linestyle="--", label="BB Upper")
axes[0].plot(idx, bb_middle, linewidth=0.5, color="#9E9E9E", linestyle=":", label="BB Middle")
axes[0].plot(idx, bb_lower, linewidth=0.5, color="#4CAF50", linestyle="--", label="BB Lower")
axes[0].fill_between(idx, bb_lower, bb_upper, alpha=0.08, color="#9C27B0")
axes[0].set_ylabel("Price")
axes[0].legend(loc="upper left", fontsize=8)
axes[0].set_title("Technical Indicators")

# RSI
axes[1].plot(idx, rsi, linewidth=0.8, color="#FF9800")
axes[1].axhline(70, color="red", linewidth=0.5, linestyle="--")
axes[1].axhline(30, color="green", linewidth=0.5, linestyle="--")
axes[1].fill_between(idx, 30, 70, alpha=0.05, color="grey")
axes[1].set_ylabel("RSI(14)")
axes[1].set_ylim(10, 90)

# MACD
axes[2].plot(idx, macd_line, linewidth=0.7, color="#2196F3", label="MACD")
axes[2].plot(idx, macd_signal, linewidth=0.7, color="#E91E63", label="Signal")
colours = ["#4CAF50" if v >= 0 else "#F44336" for v in macd_hist]
axes[2].bar(idx, macd_hist, width=0.0005, color=colours, alpha=0.6)
axes[2].set_ylabel("MACD(12,26,9)")
axes[2].legend(loc="upper left", fontsize=8)

# ATR
axes[3].plot(idx, atr, linewidth=0.8, color="#9C27B0")
axes[3].set_ylabel("ATR(14)")
axes[3].set_xlabel("Time")

plt.tight_layout()
plt.show()

## 3. Volatility Regime Detection (HMM)

In [ ]:
from qts.models.base import VolLevel
from qts.signals.regime import RegimeDetector

detector = RegimeDetector(random_state=SEED)
detector.fit(atr)

# Classify each bar from the point sufficient history exists
regimes = []
confidences = []
for i in range(len(bars)):
    atr_slice = atr[: i + 1]
    valid = atr_slice[~np.isnan(atr_slice)]
    if len(valid) < 10:
        regimes.append(None)
        confidences.append(np.nan)
    else:
        regime, conf = detector.predict(atr_slice)
        regimes.append(regime)
        confidences.append(conf)

df["regime"] = regimes
df["regime_confidence"] = confidences

n_high = sum(1 for r in regimes if r == VolLevel.HIGH)
n_low = sum(1 for r in regimes if r == VolLevel.LOW)
print(f"Regime split: HIGH={n_high}, LOW={n_low}, warmup={sum(1 for r in regimes if r is None)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Price coloured by regime
ax1.plot(idx, closes, linewidth=0.6, color="#9E9E9E", alpha=0.5)
for i in range(len(bars)):
    if regimes[i] == VolLevel.HIGH:
        ax1.axvspan(idx[max(0, i - 1)], idx[i], alpha=0.15, color="red")
ax1.set_ylabel("Price")
ax1.set_title("Volatility Regime Overlay (red = HIGH)")

# Confidence
ax2.fill_between(idx, confidences, alpha=0.4, color="#673AB7")
ax2.set_ylabel("Regime Confidence")
ax2.set_ylim(0, 1)
ax2.set_xlabel("Time")

plt.tight_layout()
plt.show()

## 4. Sentiment Analysis & Fusion

In [ ]:
from qts.nlp.fusion import SentimentFusion
from qts.nlp.vader import VaderAnalyzer

headlines = [
    "Bitcoin surges past $50K as institutional adoption grows",
    "Crypto market faces regulatory headwinds in EU",
    "BTC mining difficulty hits all-time high",
    "Major bank announces crypto custody service",
    "Whale wallet moves 10,000 BTC to exchange",
    "Layer 2 scaling solution sees record TVL growth",
    "SEC delays Bitcoin ETF decision again",
    "El Salvador reports profit on national BTC holdings",
]

analyzer = VaderAnalyzer()
results = analyzer.analyze(headlines)

sent_df = pd.DataFrame([
    {"headline": r.text[:60], "label": r.label, "score": r.score, "confidence": r.confidence}
    for r in results
])
sent_df

In [ ]:
# Compute directional scores and fuse
directional = [
    r.score * (1 if r.label == "POSITIVE" else -1 if r.label == "NEGATIVE" else 0)
    for r in results
]
avg_news_score = sum(directional) / len(directional)

fusion = SentimentFusion()
fused = fusion.fuse(news_score=avg_news_score, social_score=0.05, geopolitical_score=0.0)

print(f"Average news sentiment: {avg_news_score:+.3f}")
print(f"Fused sentiment (news={fusion.weights.news}, social={fusion.weights.social}, geo={fusion.weights.geopolitical}): {fused:+.3f}")

## 5. Signal Pipeline & Combined Alpha

In [ ]:
from qts.config import SentimentFusionWeights, SignalWeights, StrategyParams
from qts.signals.alpha import combined_alpha
from qts.signals.pipeline import SignalPipeline

params = StrategyParams(
    version="demo-v1",
    weights=SignalWeights(w_rsi=0.20, w_macd=0.20, w_bb=0.15, w_mom=0.15, w_sentiment=0.30),
    entry_threshold=0.25,
    exit_threshold=-0.10,
    max_hold_bars=48,
    sentiment_fusion_weights=SentimentFusionWeights(news=0.4, social=0.3, geopolitical=0.3),
)

pipeline = SignalPipeline(symbol=SYMBOL, sentiment_score=fused)

# Compute snapshots for each bar (after warmup)
snapshots = []
alphas = []
for i in range(len(bars)):
    snap = pipeline.compute(bars[: i + 1])
    if snap is not None:
        alpha = combined_alpha(snap, params)
        snapshots.append(snap)
        alphas.append(alpha)

print(f"{len(snapshots)} signal snapshots computed ({len(bars) - len(snapshots)} warmup bars skipped)")
print(f"Latest snapshot:")
print(f"  RSI={snapshots[-1].rsi:.1f}  MACD_hist={snapshots[-1].macd_histogram:.4f}")
print(f"  BB_pos={snapshots[-1].bb_position:.3f}  ATR={snapshots[-1].atr:.2f}")
print(f"  Regime={snapshots[-1].vol_level.value} (conf={snapshots[-1].vol_level_confidence:.2f})")
print(f"  Sentiment={snapshots[-1].sentiment_score:+.3f}")
print(f"  Combined Alpha={alphas[-1]:+.4f}")

In [ ]:
# Alpha time series
alpha_idx = idx[len(bars) - len(alphas) :]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(idx, closes, linewidth=0.8, color="#2196F3")
ax1.set_ylabel("Price")
ax1.set_title("Price vs Combined Alpha")

alpha_arr = np.array(alphas)
ax2.fill_between(alpha_idx, alpha_arr, 0, where=alpha_arr >= 0,
                  color="#4CAF50", alpha=0.5, label="Bullish")
ax2.fill_between(alpha_idx, alpha_arr, 0, where=alpha_arr < 0,
                  color="#F44336", alpha=0.5, label="Bearish")
ax2.axhline(params.entry_threshold, color="green", linewidth=0.5, linestyle="--", label="Entry threshold")
ax2.axhline(-params.entry_threshold, color="red", linewidth=0.5, linestyle="--")
ax2.set_ylabel("Combined Alpha")
ax2.set_xlabel("Time")
ax2.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Backtest — Strategy Comparison

In [ ]:
from qts.config import get_settings
from qts.simulation.backtest import BacktestEngine, BacktestSettings
from qts.strategies.mean_reversion import MeanReversionStrategy
from qts.strategies.momentum import MomentumStrategy
from qts.strategies.sma_crossover import SMACrossoverStrategy

get_settings.cache_clear()
settings = get_settings()

INITIAL_CAPITAL = 100_000.0
bt_settings = BacktestSettings(initial_capital=INITIAL_CAPITAL)

strategies = {
    "SMA Crossover": SMACrossoverStrategy(fast_period=10, slow_period=30, quantity=1.0),
    "Momentum": MomentumStrategy(settings.strategy, settings.risk, portfolio_value=INITIAL_CAPITAL),
    "Mean Reversion": MeanReversionStrategy(portfolio_value=INITIAL_CAPITAL),
}

bt_results = {}
for name, strategy in strategies.items():
    engine = BacktestEngine(strategy, bars, bt_settings)
    bt_results[name] = engine.run()

# Summary table
summary = pd.DataFrame({
    name: {
        "Sharpe": f"{r.sharpe_ratio:.2f}",
        "Return": f"{r.total_return * 100:+.2f}%",
        "Max DD": f"{r.max_drawdown * 100:.2f}%",
        "Win Rate": f"{r.win_rate * 100:.1f}%",
        "Profit Factor": f"{r.profit_factor:.2f}",
        "Trades": len(r.trades),
    }
    for name, r in bt_results.items()
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colours = ["#2196F3", "#FF9800", "#4CAF50"]
for (name, result), colour in zip(bt_results.items(), colours):
    eq = np.array(result.equity_curve)
    ax.plot(eq, linewidth=1.0, color=colour, label=f"{name} (Sharpe {result.sharpe_ratio:.2f})")

ax.axhline(INITIAL_CAPITAL, color="grey", linewidth=0.5, linestyle="--")
ax.set_xlabel("Bar")
ax.set_ylabel("Portfolio Value (USD)")
ax.set_title("Equity Curves — Strategy Comparison")
ax.legend()

plt.tight_layout()
plt.show()

## 7. Risk Controls

In [ ]:
from qts.execution.risk import RiskManager

risk_mgr = RiskManager(settings.risk)

checks = {
    "Position size (1% of portfolio)": risk_mgr.check_position_size(
        INITIAL_CAPITAL * 0.01, INITIAL_CAPITAL
    ),
    "Daily drawdown (-0.5%)": risk_mgr.check_daily_drawdown(-500.0, INITIAL_CAPITAL),
    "Circuit breaker": not risk_mgr.is_halted(),
}

print("Risk Limits:")
print(f"  Max daily drawdown: {settings.risk.max_daily_drawdown_pct:.0%}")
print(f"  Max position size:  {settings.risk.max_position_size_pct:.0%}")
print(f"  Max open positions: {settings.risk.max_open_positions}")
print(f"  Circuit breaker cooldown: {settings.risk.circuit_breaker_cooldown_seconds}s")
print()
print("Pre-trade checks:")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {check}: {status}")

## Summary

This notebook demonstrated the full QTS data flow:

```
Market Data → Technical Indicators → Regime Detection → Sentiment Fusion
                              ↓
                      Combined Alpha → Strategy Decisions → Backtest
                              ↓
                      Risk Manager (pre-trade validation)
```

All components run on synthetic data with deterministic seeds for reproducibility.